# Investing.com Feature Engineering

The daily tickers are shifted by 24 hours, so day D sees the settled price of D-1. The columns are unknown in the `TimeSeriesDataSet`. `create_all_features` builds the lagged levels.

The TFT feature set grows in three steps. Investing.com belongs to D1 only (fuels, ETS and currency), so one file is saved: `d1`. `select_features(df, model, step)` keeps it: `model` is `"tft"` and `step` is `"d1"`.

**TFT features (D1)**

* `ttf_lag24`, `eua_lag24`, `coal_lag24`, `eurpln_lag24` — price levels shifted by 24 hours.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_merged_data,
    save_merged_data,
    inspect_merged_file,
)
from src.utils.feature_engineering.fe_investing import (
    COLUMNS,
    create_all_features,
    select_features,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

data_inv = load_merged_data(
    DATA_DIR / "preprocessed" / "investing_com" / date_range / "investing_com.parquet"
)

## 1. Input Columns

The preprocessed prices, before the lag and the return features.

In [3]:
print(list(data_inv.columns))

['dutch_ttf_natural_gas_futures', 'eu_carbon_emissions_futures', 'eur_pln', 'rotterdam_coal_futures']


## 2. Features

Lagged levels `ttf_lag24`, `eua_lag24`, `coal_lag24` and `eurpln_lag24`.

In [4]:
features = create_all_features(data_inv)

In [5]:
print(features.shape)
print(list(features.columns))

(38473, 4)
['ttf_lag24', 'eua_lag24', 'coal_lag24', 'eurpln_lag24']


## 3. The TFT Step

`COLUMNS["tft"]["d1"]` keeps the four lagged levels.

In [6]:
for model, steps in COLUMNS.items():
    for step in steps:
        subset = select_features(features, model, step)
        print(f"=== {model}/{step} ===")
        print(subset.shape[1], "features")
        print(list(subset.columns))

=== tft/d1 ===
4 features
['ttf_lag24', 'eua_lag24', 'coal_lag24', 'eurpln_lag24']


## 4. Conclusions

The D1 columns are saved to `data/features/investing_com/tft/d1/`, and the file is verified by reading it back.

The file covers 2022-06-01 to 2026-10-21 (UTC). The only missing values are the first 24 hours, which are needed for the 24-hour lag (the first complete hour is 2022-06-02 00:00 UTC).

## 5. Saving Feature Data

In [7]:
for model, steps in COLUMNS.items():
    for step in steps:
        path = (
            DATA_DIR
            / "features"
            / "investing_com"
            / model
            / step
            / f"{date_range}.parquet"
        )
        save_merged_data(select_features(features, model, step), path)
        inspect_merged_file(path)

  [FILE] data/features/investing_com/tft/d1/2022-06-09_2026-10-21.parquet | shape: (38473, 4) | idx: 2022-06-01 00:00:00+00:00-2026-10-21 00:00:00+00:00 | cols: 4 | NaN: 96
